> **日本語版について**: このノートブックは、[jpmorganchase/python-training](https://github.com/jpmorganchase/python-training) の英語原文を日本語に翻訳したものです（Apache License 2.0）。コードの動作は原文から変更していません。

# Web サイトからデータを取得する
Web サイトによっては、データを取り出しにくいものもあります。そんなときは `pandas` が役立ちます。

In [1]:
import pandas as pd
import requests
from bs4 import BeautifulSoup

result = requests.get('https://en.wikipedia.org/wiki/List_of_sovereign_states')
pd.read_html(result.content)[0].head(20)

,通称と正式名称,国連システムにおける地位[a],主権をめぐる争い[b],地位と主権の承認に関する補足情報[d]
0,NaN,NaN,NaN,NaN
1,国連加盟国とオブザーバー国 ↓,NaN,NaN,NaN
2,アブハジア,NaN,NaN,NaN
3,アフガニスタン – アフガニスタン・イスラム共和国,国連加盟国,NaN,NaN
4,アルバニア – アルバニア共和国,NaN,NaN,NaN
5,アルジェリア – アルジェリア人民民主共和国,NaN,NaN,NaN
6,アンドラ – アンドラ公国,NaN,NaN,Andorra is a co-principality in which the offi...
7,アンゴラ – アンゴラ共和国,NaN,NaN,NaN
8,アンティグア・バーブーダ,NaN,NaN,アンティグア・バーブーダは英連邦王国である[e]...
9,アルゼンチン – アルゼンチン共和国[g],NaN,NaN,Argentina is a federation of 23 provinces and ...


もっと複雑な解析には `BeautifulSoup` を使います。同じ表を、今度はこのライブラリで抽出してみましょう。

In [2]:
soup = BeautifulSoup(result.content, 'lxml') # HTML を文字列として解析
str(soup)[:500]

'<!DOCTYPE html>\n<html class="client-nojs" dir="ltr" lang="en">\n<head>\n<meta charset="utf-8"/>\n<title>List of sovereign states - Wikipedia</title>\n<script>document.documentElement.className="client-js";RLCONF={"wgBreakFrames":!1,"wgSeparatorTransformTable":["",""],"wgDigitTransformTable":["",""],"wgDefaultDateFormat":"dmy","wgMonthNames":["","January","February","March","April","May","June","July","August","September","October","November","December"],"wgRequestId":"XqsP3wpAAEUAAEGqjmIAAAAR","wgCS'

表を探します。

In [3]:
tables = soup.find_all('table')

`pandas` の `read_html` で、最初の表を DataFrame に読み込みます。

In [4]:
pd.read_html(str(tables[0]))[0].head(20)

,通称と正式名称,国連システムにおける地位[a],主権をめぐる争い[b],地位と主権の承認に関する補足情報[d]
0,NaN,NaN,NaN,NaN
1,国連加盟国とオブザーバー国 ↓,NaN,NaN,NaN
2,アブハジア,NaN,NaN,NaN
3,アフガニスタン – アフガニスタン・イスラム共和国,国連加盟国,NaN,NaN
4,アルバニア – アルバニア共和国,NaN,NaN,NaN
5,アルジェリア – アルジェリア人民民主共和国,NaN,NaN,NaN
6,アンドラ – アンドラ公国,NaN,NaN,Andorra is a co-principality in which the offi...
7,アンゴラ – アンゴラ共和国,NaN,NaN,NaN
8,アンティグア・バーブーダ,NaN,NaN,アンティグア・バーブーダは英連邦王国である[e]...
9,アルゼンチン – アルゼンチン共和国[g],NaN,NaN,Argentina is a federation of 23 provinces and ...


見てのとおり、返ってくるデータはいつもきれいとは限りません。HTML を解析するより API のほうが楽なのは、このためです。

とはいえ、この手順を関数にまとめれば、かなり楽になります。

In [5]:
def dfFromURL(url, tableNumber=1):
    soup = BeautifulSoup(requests.get(url).content, 'lxml') # HTML を文字列として解析
    tables = soup.find_all('table')
    # 指定した番号の表がページにあるか確認
    assert len(tables) >= tableNumber
    return pd.read_html(str(tables[tableNumber-1]))[0]

これで、HTML の表を DataFrame として取得できます。呼び出しは簡単です。試してみましょう。

In [6]:
prices = dfFromURL('https://finance.yahoo.com/quote/JPM/history?p=JPM')
prices.head()

,日付,始値,高値,安値,終値※,調整後終値※※,出来高
0,2020年4月30日,95.60,96.84,94.55,95.76,95.76,17262125
1,2020年4月29日,99.08,99.45,97.27,97.86,97.86,20427700
2,2020年4月28日,97.81,98.36,95.03,95.29,95.29,20238900
3,2020年4月27日,91.94,95.63,91.54,94.62,94.62,22870200
4,2020年4月24日,90.64,91.10,88.49,90.71,90.71,15815300


div や末尾の免責事項が混ざったデータが返ってきました。`dropna` で簡単に取り除きましょう。

In [7]:
prices = prices.dropna()
prices.head()

,日付,始値,高値,安値,終値※,調整後終値※※,出来高
0,2020年4月30日,95.60,96.84,94.55,95.76,95.76,17262125
1,2020年4月29日,99.08,99.45,97.27,97.86,97.86,20427700
2,2020年4月28日,97.81,98.36,95.03,95.29,95.29,20238900
3,2020年4月27日,91.94,95.63,91.54,94.62,94.62,22870200
4,2020年4月24日,90.64,91.10,88.49,90.71,90.71,15815300


次に、Web サイトから2番目の表を取得してみましょう。キャバリアーズ（キャブズ）の直近数シーズンの成績を見てみます：

In [17]:
df1 = dfFromURL('https://www.espn.com/nba/team/stats/_/name/cle', 0)
df1

,FGM,FGA,FG%,3PM,3PA,3P%,FTM,FTA,FT%,2PM,2PA,2P%,SC-EFF,SH-EFF
0,7.9,16.7,47.2,1.5,3.9,38.0,3.5,4.1,84.6,6.4,12.8,50.1,1.242,0.52
1,5.8,13.0,45.0,2.6,7.0,37.4,3.3,3.9,85.4,3.2,6.0,53.7,1.358,0.55
2,7.3,13.1,55.2,0.5,1.8,28.6,2.5,4.9,51.3,6.8,11.4,59.3,1.333,0.57
3,5.0,11.2,44.2,2.0,5.5,37.1,2.6,3.0,88.4,2.9,5.8,50.9,1.298,0.53
4,4.7,11.8,40.1,1.8,5.0,35.5,1.1,1.2,87.5,3.0,6.8,43.4,1.041,0.48
5,5.1,9.9,51.2,0.2,0.4,39.1,1.7,2.8,61.5,4.9,9.5,51.8,1.217,0.52
6,4.1,9.4,43.7,1.9,4.9,38.3,1.0,1.4,67.0,2.2,4.4,49.7,1.179,0.54
7,4.1,7.7,53.1,1.0,2.8,35.2,0.9,1.3,67.6,3.1,4.9,63.6,1.309,0.60
8,3.7,8.4,44.2,1.1,3.2,33.5,1.5,2.0,72.3,2.7,5.3,50.6,1.183,0.50
9,1.9,4.0,47.9,0.5,1.5,35.1,1.3,1.7,73.2,1.4,2.5,55.9,1.406,0.55


In [18]:
df2 = dfFromURL('https://www.espn.com/nba/team/stats/_/name/cle', 1)
df2

,名前
0,Collin Sexton PG
1,Kevin Love PF
2,Andre Drummond C *
3,Jordan Clarkson PG *
4,Darius Garland PG
5,Tristan Thompson C
6,Cedi Osman SF
7,Larry Nance Jr. PF
8,Kevin Porter Jr. SG
9,Dante Exum PG *


In [21]:
pd.concat((df2, df1), axis=1)

,名前,FGM,FGA,FG%,3PM,3PA,3P%,FTM,FTA,FT%,2PM,2PA,2P%,SC-EFF,SH-EFF
0,Collin Sexton PG,7.9,16.7,47.2,1.5,3.9,38.0,3.5,4.1,84.6,6.4,12.8,50.1,1.242,0.52
1,Kevin Love PF,5.8,13.0,45.0,2.6,7.0,37.4,3.3,3.9,85.4,3.2,6.0,53.7,1.358,0.55
2,Andre Drummond C *,7.3,13.1,55.2,0.5,1.8,28.6,2.5,4.9,51.3,6.8,11.4,59.3,1.333,0.57
3,Jordan Clarkson PG *,5.0,11.2,44.2,2.0,5.5,37.1,2.6,3.0,88.4,2.9,5.8,50.9,1.298,0.53
4,Darius Garland PG,4.7,11.8,40.1,1.8,5.0,35.5,1.1,1.2,87.5,3.0,6.8,43.4,1.041,0.48
5,Tristan Thompson C,5.1,9.9,51.2,0.2,0.4,39.1,1.7,2.8,61.5,4.9,9.5,51.8,1.217,0.52
6,Cedi Osman SF,4.1,9.4,43.7,1.9,4.9,38.3,1.0,1.4,67.0,2.2,4.4,49.7,1.179,0.54
7,Larry Nance Jr. PF,4.1,7.7,53.1,1.0,2.8,35.2,0.9,1.3,67.6,3.1,4.9,63.6,1.309,0.60
8,Kevin Porter Jr. SG,3.7,8.4,44.2,1.1,3.2,33.5,1.5,2.0,72.3,2.7,5.3,50.6,1.183,0.50
9,Dante Exum PG *,1.9,4.0,47.9,0.5,1.5,35.1,1.3,1.7,73.2,1.4,2.5,55.9,1.406,0.55
